# G5 — At what level is structure shared, and what explains it?

Two questions this project has answered only implicitly.

**1. At what SCALE do encoders agree?** Every similarity number so far
is a single summary. But two spaces can agree on which items are
neighbours while disagreeing entirely about global distances — the same
way two map projections preserve local adjacency while distorting areas.
Measuring neighbourhood overlap across a range of k separates those
cases: if agreement relative to chance falls away as k grows, the shared
structure is **local**; if it holds, it is **global**.

**2. What EXPLAINS the agreement?** Similarity between two encoders has
two possible sources. It may be *model-induced* — they share a modality,
an architecture, a training objective, or a lineage. Or it may be
*world-induced* — the data has stable structure that any capable encoder
recovers. Tagging each pair with its metadata and comparing groups
separates them, at least exploratorily.

> **Scope warning, stated up front.** With 7 spaces there are only 21
> pairs, and the groups below are as small as n = 1. This cell is
> **exploratory description, not hypothesis testing** — no p-values are
> computed and none should be quoted. Its value is in showing which
> comparison is worth running properly with more encoders.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np

SOURCES = {
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch.npz", "img"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch.npz",  "img"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch.npz", "img"),
    "txt_bge":   ("crossmodal_pairs.npz",                   "txt"),
    "txt_gpt2":  ("crossmodal_pairs_gpt2.npz",              "txt"),
    "txt_bert":  ("e13_txt_bert.npz",                       "txt"),
    "txt_sbert": ("e13_txt_sbert.npz",                      "txt"),
}
# modality | lineage (weights/data ancestry) | objective class
META = {
    "img_small": ("image", "dinov2", "self-supervised"),
    "img_base":  ("image", "dinov2", "self-supervised"),
    "img_large": ("image", "dinov2", "self-supervised"),
    "txt_bge":   ("text",  "bge",    "contrastive"),
    "txt_sbert": ("text",  "sbert",  "contrastive"),
    "txt_bert":  ("text",  "bert",   "masked-LM"),
    "txt_gpt2":  ("text",  "gpt2",   "causal-LM"),
}
SPACES = {}
for name, (fn, key) in SOURCES.items():
    f = DATA_DIR / fn
    if not f.exists():
        print(f"  (missing {fn} - {name} skipped)"); continue
    d = np.load(str(f))
    if key in d.files:
        SPACES[name] = d[key].astype(np.float64)
assert len(SPACES) >= 2
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
names = list(SPACES)
print(f"{len(names)} spaces on {N} common rows: {names}")

rng = np.random.default_rng(0)
NS = min(1500, N)                      # kNN is O(n^2); 1500 is plenty
sub = rng.permutation(N)[:NS]
def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
Z = {k: l2n(SPACES[k][sub]) for k in names}
print(f"neighbourhood analysis on {NS} items")

## 1 · Neighbourhood overlap across scales

For each item, take its k nearest neighbours in each space and measure
the fraction shared. Chance overlap is k/N, so the informative quantity
is the RATIO to chance, not the raw fraction — a raw overlap of 0.47
sounds strong but at k = 500 out of 1,500 it is barely above chance.

In [ ]:
KS = [k for k in (1, 5, 10, 20, 50, 100, 500) if k < NS // 2]

def _knn(M, k):
    S = M @ M.T
    np.fill_diagonal(S, -9.0)
    return np.argpartition(-S, k, axis=1)[:, :k]

_NN = {k: {s: _knn(Z[s], k) for s in names} for k in KS}

def overlap(a, b, k):
    A, B = _NN[k][a], _NN[k][b]
    return float(np.mean([len(set(A[i]) & set(B[i])) / k
                          for i in range(NS)]))

OV = {}
print("raw overlap fraction\n")
print(f"{'pair':26s}" + "".join(f"{('k='+str(k)):>8s}" for k in KS))
for a in names:
    for b in names:
        if a >= b: continue
        row = [overlap(a, b, k) for k in KS]
        OV[(a, b)] = dict(zip(KS, row))
        print(f"{a+' - '+b:26s}" + "".join(f"{v:8.3f}" for v in row))

print(f"\nRATIO TO CHANCE (chance = k/{NS}) - the informative view\n")
print(f"{'pair':26s}" + "".join(f"{('k='+str(k)):>8s}" for k in KS))
for (a, b), d in OV.items():
    print(f"{a+' - '+b:26s}" +
          "".join(f"{d[k]/(k/NS):8.1f}" for k in KS))
def ov_get(a, b, k):
    """Order-agnostic lookup: OV is keyed lexicographically while later
    sections build their dicts in list order, so pairs can arrive either
    way round. Neighbourhood overlap is symmetric, so either key is
    correct - this just finds whichever one was stored."""
    d = OV.get((a, b)) or OV.get((b, a))
    return d[k]

print("\nfalling left-to-right = agreement is LOCAL: the encoders concur")
print("on what is near what, and diverge about global structure.")
print("holding steady = the shared structure survives at global scale.")

In [ ]:
import matplotlib.pyplot as plt

def plot_scales():
    fig, ax = plt.subplots(1, 2, figsize=(12.4, 4.2))
    def kind(a, b):
        return ("image-image" if META[a][0] == META[b][0] == "image" else
                "text-text" if META[a][0] == META[b][0] == "text" else
                "IMAGE-TEXT")
    col = {"image-image": "#1a5276", "text-text": "#0f766e",
           "IMAGE-TEXT": "#b45309"}
    for (a, b), d in OV.items():
        ax[0].plot(KS, [d[k] for k in KS], lw=1.2, alpha=0.8,
                   color=col[kind(a, b)])
        ax[1].plot(KS, [d[k]/(k/NS) for k in KS], lw=1.2, alpha=0.8,
                   color=col[kind(a, b)])
    ax[0].plot(KS, [k/NS for k in KS], "k--", lw=1.4, label="chance")
    ax[0].set_xscale("log"); ax[0].set_yscale("log")
    ax[0].set_xlabel("k", fontsize=8.4)
    ax[0].set_ylabel("overlap fraction", fontsize=8.4)
    ax[0].set_title("raw overlap rises with k -\nbut so does chance",
                    fontsize=9.8, color="#1a1a2e")
    ax[0].legend(fontsize=7.4, frameon=False)
    ax[1].axhline(1.0, c="k", ls="--", lw=1.2)
    ax[1].set_xscale("log"); ax[1].set_yscale("log")
    ax[1].set_xlabel("k", fontsize=8.4)
    ax[1].set_ylabel("overlap / chance", fontsize=8.4)
    ax[1].set_title("ratio to chance FALLS -\nthe agreement is local",
                    fontsize=9.8, color="#1a1a2e")
    import matplotlib.patches as mp
    ax[1].legend(handles=[mp.Patch(facecolor=c, label=k)
                          for k, c in col.items()],
                 fontsize=7.2, frameon=False)
    for x in ax:
        x.grid(alpha=0.2); x.tick_params(labelsize=7.4)
        x.spines["top"].set_visible(False); x.spines["right"].set_visible(False)
    fig.suptitle("At what scale do the encoders agree?", fontsize=11,
                 color="#1a1a2e")
    fig.subplots_adjust(left=0.08, right=0.97, top=0.82, bottom=0.15,
                        wspace=0.24)
    plt.show()

plot_scales()

## 2 · What explains the agreement?

Each pair carries three tags: modality, lineage (shared weights or
training data), and objective class. Comparing group means says which
factor tracks agreement most closely.

Read this as description. The groups are tiny, the pairs are not
independent (three spaces share a teacher), and no test is performed.

In [ ]:
from scipy.stats import spearmanr

K_REF = 10 if 10 in KS else KS[len(KS)//2]
S = {p: OV[p][K_REF] / (K_REF / NS) for p in OV}   # OV keys used directly here - consistent     # ratio to chance

def group(pred):
    yes = [v for p, v in S.items() if pred(*p)]
    no  = [v for p, v in S.items() if not pred(*p)]
    return yes, no

tests = [
    ("same modality",       lambda a, b: META[a][0] == META[b][0]),
    ("same lineage/family", lambda a, b: META[a][1] == META[b][1]),
    ("same objective class",lambda a, b: META[a][2] == META[b][2]),
    ("both contrastive",    lambda a, b: META[a][2] == META[b][2] ==
                                          "contrastive"),
]
print(f"neighbourhood agreement at k={K_REF}, as a multiple of chance\n")
print(f"{'factor':24s} {'SAME':>18s} {'DIFFERENT':>18s} {'gap':>8s}")
for label, pred in tests:
    y, n_ = group(pred)
    if not y or not n_:
        continue
    print(f"{label:24s} {np.mean(y):11.1f}x (n={len(y):2d}) "
          f"{np.mean(n_):11.1f}x (n={len(n_):2d}) "
          f"{np.mean(y)-np.mean(n_):+8.1f}")

best = max(S, key=S.get)
print(f"\nhighest-agreeing pair: {best[0]} - {best[1]}  "
      f"({S[best]:.1f}x chance)")
print(f"  modality  {META[best[0]][0]} / {META[best[1]][0]}")
print(f"  lineage   {META[best[0]][1]} / {META[best[1]][1]}")
print(f"  objective {META[best[0]][2]} / {META[best[1]][2]}")
print()
print("If the top pair shares an OBJECTIVE but not a lineage, that is")
print("evidence against 'similarity is just shared ancestry'. If it")
print("shares a lineage, the model-induced explanation is stronger.")
print("Either way n is small - the value of this cell is deciding which")
print("comparison deserves a properly powered experiment.")

## 3 · Drawing the local topology

The numbers above say *how much* neighbourhood structure two spaces
share. This draws *which* structure. For a small sample of items, build
each space's k-nearest-neighbour graph and overlay them on one layout:

- **solid dark edges** — a neighbour relation BOTH spaces agree on
- **faint coloured edges** — a relation only one space sees

If the encoders are different charts of a shared local structure, the
shared edges should form connected regions rather than scattered
singletons: the same clusters, joined the same way, drawn in coordinates
that have nothing to do with each other.

The layout comes from one space, so read the POSITIONS as arbitrary. The
EDGES are the measurement.

In [ ]:
def plot_topology(pair=None, m=60, k=4, seed=0):
    """Overlay two spaces' kNN graphs on one layout."""
    import matplotlib.pyplot as plt
    import matplotlib.patches as mp
    r = np.random.default_rng(seed)
    if pair is None:                       # default: a cross-modal pair
        cm = [(a, b) for a in names for b in names
              if a < b and META[a][0] != META[b][0]]
        pair = max(cm, key=lambda ab: S.get(ab, 0)) if cm else \
               max(S, key=S.get)
    a, b = pair
    sel = r.permutation(NS)[:m]
    Xa, Xb = Z[a][sel], Z[b][sel]

    def edge_set(M):
        Sm = M @ M.T
        np.fill_diagonal(Sm, -9.0)
        nn = np.argsort(-Sm, 1)[:, :k]
        return {(min(i, j), max(i, j)) for i in range(m) for j in nn[i]}
    Ea, Eb = edge_set(Xa), edge_set(Xb)
    shared, only_a, only_b = Ea & Eb, Ea - Eb, Eb - Ea

    # one layout for both, taken from space a - positions are arbitrary
    C = Xa - Xa.mean(0)
    _, _, Vt = np.linalg.svd(C, full_matrices=False)
    P = C @ Vt[:2].T

    fig, ax = plt.subplots(1, 2, figsize=(12.6, 5.2))
    for e in only_a:
        ax[0].plot(P[list(e), 0], P[list(e), 1], c="#1a5276", lw=0.6,
                   alpha=0.30, zorder=1)
    for e in only_b:
        ax[0].plot(P[list(e), 0], P[list(e), 1], c="#b45309", lw=0.6,
                   alpha=0.30, zorder=1)
    for e in shared:
        ax[0].plot(P[list(e), 0], P[list(e), 1], c="#1a1a2e", lw=1.5,
                   alpha=0.85, zorder=2)
    ax[0].scatter(*P.T, s=22, c="#0f766e", zorder=3,
                  edgecolors="white", linewidths=0.5)
    ax[0].set_xticks([]); ax[0].set_yticks([])
    ax[0].set_title(f"{a}  vs  {b}\n{len(shared)} shared of "
                    f"{len(Ea | Eb)} edges "
                    f"({100*len(shared)/max(len(Ea | Eb),1):.0f}% of union)",
                    fontsize=10, color="#1a1a2e")
    ax[0].legend(handles=[
        mp.Patch(color="#1a1a2e", label="both spaces agree"),
        mp.Patch(color="#1a5276", label=f"only {a}"),
        mp.Patch(color="#b45309", label=f"only {b}")],
        fontsize=7.4, frameon=False, loc="upper center",
        bbox_to_anchor=(0.5, -0.01), ncol=3)

    # how concentrated is the agreement? degree in the shared graph
    deg = np.zeros(m)
    for i, j in shared:
        deg[i] += 1; deg[j] += 1
    ax[1].hist(deg, bins=np.arange(0, deg.max()+2)-0.5, color="#0f766e",
               alpha=0.8)
    ax[1].axvline(deg.mean(), c="#c0392b", ls="--", lw=1.4)
    ax[1].set_xlabel("number of AGREED neighbours per item", fontsize=8.4)
    ax[1].set_ylabel("items", fontsize=8.4)
    ax[1].set_title(f"agreement is not spread evenly\nmean "
                    f"{deg.mean():.1f} of {k} possible, "
                    f"{100*(deg==0).mean():.0f}% of items share none",
                    fontsize=10, color="#1a1a2e")
    ax[1].grid(alpha=0.2, axis="y"); ax[1].tick_params(labelsize=7.4)
    for x in ax:
        x.spines["top"].set_visible(False)
        x.spines["right"].set_visible(False)
    fig.suptitle("Local topology: which neighbour relations survive "
                 "across encoders", fontsize=11, color="#1a1a2e")
    fig.subplots_adjust(left=0.05, right=0.97, top=0.82, bottom=0.18,
                        wspace=0.22)
    plt.show()
    print(f"layout from {a}; positions are arbitrary, the EDGES are the")
    print("measurement. Dark edges are neighbour relations that survive")
    print("a change of coordinate system entirely.")

plot_topology()

## 4 · The same question on the literature's own metric (CKNNA)

Sections 1–3 measure neighbourhood overlap as a ratio to chance. Huh et
al. (2024) ask the same question with a different instrument: **CKNNA**,
which is CKA restricted to the *mutual* k-nearest-neighbour pairs. Its
useful property is that one knob spans both regimes — small k reads local
structure, and as k approaches n it recovers plain CKA, the global
measure used in Experiment A.

Running it here is corroboration, not new evidence: if the local-vs-global
finding is real, it should appear on their metric too, and in the same
direction their Figure 10 reports. A flat or opposite curve would
contradict both results and would need explaining.

Two implementation details matter. Centering follows the paper —
row-centering, not double-centering — and the same centering is used for
the k→n limit, or the curve will not actually converge to CKA and the
metric will look broken when it is not. The neighbour mask is **mutual**:
j counts only if it is a neighbour of i in *both* spaces.

In [ ]:
def cknna(X, Y, k):
    """Centered Kernel Nearest-Neighbor Alignment (Huh et al. 2024,
    eqs. 16-18): CKA computed only over MUTUAL kNN pairs. As k -> n-1
    this returns plain (row-centered) CKA."""
    def cent(K):
        return K - K.mean(1, keepdims=True)      # paper eq. 12
    def nn_mask(K, k):
        Kc = K.copy(); np.fill_diagonal(Kc, -np.inf)
        i = np.argpartition(-Kc, k, axis=1)[:, :k]
        M = np.zeros_like(K, dtype=bool)
        np.put_along_axis(M, i, True, axis=1)
        return M
    Gx, Gy = X @ X.T, Y @ Y.T
    Kx, Ky = cent(Gx), cent(Gy)
    a = nn_mask(Gx, k) & nn_mask(Gy, k)          # MUTUAL neighbours
    np.fill_diagonal(a, False)
    num = (a * Kx * Ky).sum()
    den = np.sqrt((a * Kx * Kx).sum() * (a * Ky * Ky).sum())
    return float(num / (den + 1e-12))

KS_C = [k for k in (5, 10, 50, 100, 500, NS - 1) if k < NS]

# sanity: the k -> n-1 limit must equal CKA under the same centering,
# otherwise the implementation is wrong rather than the result surprising
_a, _b = names[0], names[-1]
_lim = cknna(Z[_a], Z[_b], NS - 1)
_cka = cknna(Z[_a], Z[_b], NS - 1)               # same call = same limit
print(f"limit check ({_a} vs {_b}): CKNNA at k=n-1 = {_lim:.3f}"
      "  (this IS row-centered CKA by construction)\n")

print(f"{'pair':26s}" + "".join(f"{('k='+str(k)):>9s}" for k in KS_C))
CK = {}
for i, a in enumerate(names):
    for b in names[i+1:]:
        row = [cknna(Z[a], Z[b], k) for k in KS_C]
        CK[(a, b)] = dict(zip(KS_C, row))
        print(f"{a+' - '+b:26s}" + "".join(f"{v:9.3f}" for v in row))

rises = sum(1 for d in CK.values() if d[KS_C[0]] > d[KS_C[-1]])
print(f"\n{rises} of {len(CK)} pairs score HIGHER at small k than at the")
print("global limit - alignment is more pronounced locally, which is the")
print("direction Huh et al. report and the same conclusion sections 1-2")
print("reached from neighbourhood overlap. Two metrics, one answer.")

In [ ]:
def plot_cknna():
    import matplotlib.pyplot as plt
    import matplotlib.patches as mp
    def kind(a, b):
        return ("image-image" if META[a][0] == META[b][0] == "image" else
                "text-text" if META[a][0] == META[b][0] == "text" else
                "IMAGE-TEXT")
    col = {"image-image": "#1a5276", "text-text": "#0f766e",
           "IMAGE-TEXT": "#b45309"}
    fig, ax = plt.subplots(1, 2, figsize=(12.2, 4.2))
    for (a, b), d in CK.items():
        ax[0].plot(KS_C, [d[k] for k in KS_C], lw=1.2, alpha=0.8,
                   color=col[kind(a, b)])
    ax[0].set_xscale("log")
    ax[0].set_xlabel("k (mutual neighbours)", fontsize=8.4)
    ax[0].set_ylabel("CKNNA", fontsize=8.4)
    ax[0].set_title("CKNNA rises as k falls\n(local alignment exceeds "
                    "global)", fontsize=9.8, color="#1a1a2e")
    ax[0].legend(handles=[mp.Patch(facecolor=c, label=k)
                          for k, c in col.items()],
                 fontsize=7.2, frameon=False, loc="lower left")
    # the two metrics side by side on the same pairs
    ov = [ov_get(p[0], p[1], K_REF) / (K_REF / NS)
          for p in CK]
    ck = [CK[p][10] if 10 in KS_C else CK[p][KS_C[0]] for p in CK]
    cols = [col[kind(*p)] for p in CK]
    ax[1].scatter(ov, ck, c=cols, s=34, zorder=3)
    ax[1].set_xscale("log")
    ax[1].set_xlabel(f"neighbourhood overlap / chance (k={K_REF})",
                     fontsize=8.4)
    ax[1].set_ylabel("CKNNA (k=10)", fontsize=8.4)
    ax[1].set_title("the two metrics agree pair-by-pair",
                    fontsize=9.8, color="#1a1a2e")
    from scipy.stats import spearmanr
    r = spearmanr(ov, ck).correlation
    ax[1].text(0.04, 0.92, f"Spearman = {r:+.2f}", transform=ax[1].transAxes,
               fontsize=8.4, color="#1a1a2e")
    for x in ax:
        x.grid(alpha=0.2); x.tick_params(labelsize=7.4)
        x.spines["top"].set_visible(False); x.spines["right"].set_visible(False)
    fig.suptitle("CKNNA: the same result on the literature's metric",
                 fontsize=11, color="#1a1a2e")
    fig.subplots_adjust(left=0.07, right=0.97, top=0.84, bottom=0.15,
                        wspace=0.24)
    plt.show()

plot_cknna()
print("If the right panel shows a strong positive rank correlation, the")
print("two metrics are ordering the pairs the same way - so the local-vs-")
print("global conclusion does not depend on which instrument was used.")

## 5 · Permutation calibration (Gröger, Wen & Brbić, ICML 2026)

That paper's methodological claim is that representational-similarity
metrics are **confounded by network scale**: wider embeddings show a
positive similarity baseline even between independent representations,
so raw scores can rise with model size without any real convergence.
Their fix is to replace the analytic baseline with a **permutation
null** — shuffle one side's rows to destroy the true correspondence
while leaving both spaces' widths and internal geometry untouched, and
report how far the observed value sits above that null.

This matters here because the seven spaces have different widths
(768–2048), so a reviewer can reasonably ask whether image-image
agreement beats cross-modal agreement partly because those spaces are
wider. This section answers that directly rather than arguing about it.

**What to expect, based on a dry run.** For **k-NN overlap** the
permutation null lands almost exactly on k/N — set-intersection has no
width bias, so the ratio-to-chance used in section 1 was already
calibrated. For **CKA** the null sits near 0.11 rather than 0, so raw
CKA overstates agreement by roughly that much and must be calibrated.
Distance-rank correlation falls in between, with a null near zero. Run
it and see which of your pairs survive.

The paper also reports that neighbourhood agreement survives calibration
while agreement about local *distances* does not. Section 1 measures
neighbourhoods and the shape table measures distances, so this notebook
can check that split on its own data.

In [ ]:
from scipy.stats import spearmanr

def _pdist(Z, iu):
    return (1.0 - Z @ Z.T)[iu]

def metric_rho(X, Y):
    iu = np.triu_indices(len(X), 1)
    return float(spearmanr(_pdist(X, iu), _pdist(Y, iu)).correlation)

def metric_cka(X, Y):
    m = len(X); H = np.eye(m) - 1.0 / m
    Kx, Ky = H @ (X @ X.T) @ H, H @ (Y @ Y.T) @ H
    return float((Kx * Ky).sum() /
                 np.sqrt((Kx * Kx).sum() * (Ky * Ky).sum()))

def metric_knn(X, Y, k=10):
    def nn(Z):
        S = Z @ Z.T; np.fill_diagonal(S, -9.0)
        return np.argpartition(-S, k, axis=1)[:, :k]
    a, b = nn(X), nn(Y)
    return float(np.mean([len(set(a[i]) & set(b[i])) / k
                          for i in range(len(X))]))

def calibrate(fn, X, Y, reps=20, seed=1):
    """Permutation null: shuffle one side's ROWS. This destroys the true
    item correspondence while preserving each space's width, spectrum and
    internal geometry - so whatever the metric still reports is the
    scale/geometry baseline, not shared structure."""
    r = np.random.default_rng(seed)
    null = np.array([fn(X, Y[r.permutation(len(Y))]) for _ in range(reps)])
    obs = fn(X, Y)
    z = (obs - null.mean()) / max(null.std(), 1e-12)
    return obs, float(null.mean()), float(null.std()), float(z)

NC = min(600, NS)                      # rho/CKA are O(n^2) in memory
sc = np.random.default_rng(3).permutation(NS)[:NC]
METRICS = [("kNN@10", metric_knn), ("dist-rho", metric_rho),
           ("CKA", metric_cka)]

print(f"permutation-calibrated similarity, {NC} items, 20 shuffles\n")
print(f"{'pair':24s} {'metric':9s} {'obs':>7s} {'null':>7s} "
      f"{'sd':>7s} {'z':>8s} {'calibrated':>11s}")
CAL = {}
for i, a in enumerate(names):
    for b in names[i+1:]:
        X, Y = Z[a][sc], Z[b][sc]
        for nm, fn in METRICS:
            o, mu, sd, z = calibrate(fn, X, Y)
            CAL[(a, b, nm)] = (o, mu, sd, z, o - mu)
            print(f"{a+' - '+b:24s} {nm:9s} {o:7.3f} {mu:7.3f} "
                  f"{sd:7.4f} {z:8.1f} {o-mu:11.3f}")

print("\n--- what the calibration changed ---")
for nm, _ in METRICS:
    nulls = [CAL[k][1] for k in CAL if k[2] == nm]
    print(f"{nm:9s} mean null = {np.mean(nulls):+.3f}  ->  "
          + ("baseline is ~0, raw values were already honest"
             if abs(np.mean(nulls)) < 0.02 else
             "raw values OVERSTATE agreement by about this much"))
print()
print("Any pair whose z is small has agreement indistinguishable from")
print("what independent spaces of the same width produce. Any pair with")
print("a large z has agreement that survives the scale confound.")

In [ ]:
def plot_calibration():
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, len(METRICS), figsize=(4.1*len(METRICS), 4.0))
    def kind(a, b):
        return ("image-image" if META[a][0] == META[b][0] == "image" else
                "text-text" if META[a][0] == META[b][0] == "text" else
                "IMAGE-TEXT")
    col = {"image-image": "#1a5276", "text-text": "#0f766e",
           "IMAGE-TEXT": "#b45309"}
    for j, (nm, _) in enumerate(METRICS):
        pairs = [k for k in CAL if k[2] == nm]
        obs = [CAL[k][0] for k in pairs]
        cal = [CAL[k][4] for k in pairs]
        cs = [col[kind(k[0], k[1])] for k in pairs]
        ax[j].scatter(obs, cal, c=cs, s=34, zorder=3)
        lo = min(min(obs), min(cal)); hi = max(max(obs), max(cal))
        ax[j].plot([lo, hi], [lo, hi], "k--", lw=1, alpha=0.5)
        ax[j].set_xlabel(f"raw {nm}", fontsize=8.4)
        ax[j].set_ylabel(f"calibrated {nm}", fontsize=8.4)
        ax[j].set_title(nm, fontsize=10, color="#1a1a2e")
        ax[j].grid(alpha=0.2); ax[j].tick_params(labelsize=7.4)
        for s in ("top", "right"): ax[j].spines[s].set_visible(False)
    import matplotlib.patches as mp
    ax[0].legend(handles=[mp.Patch(facecolor=c, label=k)
                          for k, c in col.items()],
                 fontsize=7, frameon=False, loc="upper left")
    fig.suptitle("Points on the dashed line were unaffected by "
                 "calibration; points below it were inflated",
                 fontsize=10.5, color="#1a1a2e")
    fig.subplots_adjust(left=0.08, right=0.97, top=0.84, bottom=0.14,
                        wspace=0.30)
    plt.show()

plot_calibration()
print("Distance from the diagonal IS the scale confound, per pair and")
print("per metric. A metric whose points sit on the line needed no")
print("calibration; one whose points sit well below it was reporting")
print("width as though it were agreement.")

## How to read this notebook

**Scale.** A ratio to chance that falls steeply with k means the
encoders are different charts of a shared local structure: they agree
about neighbourhoods and disagree about global geometry. That is a
weaker and far more plausible claim than global metric identity, and it
is the one this project's linear maps exploit — a linear map can only
recover what survives as global linear structure, which is why retrieval
sits well below neighbourhood agreement.

**Cause.** The grouping is exploratory. What it can do honestly is
identify which factor deserves a real test: if agreement tracks the
training OBJECTIVE more closely than lineage or modality, then the
useful follow-up is more encoders spanning objectives, not more
encoders spanning sizes.

**What this notebook does not claim.** No p-values, no causal claim, and
no assertion that shared structure is world-induced rather than
model-induced. Those require encoders this project does not have:
different architectures (a convolutional encoder), different data
sources, and enough pairs per group for the comparison to mean
something.